In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv('D:\\ITI\\9 Months Training AI\\NLP\\Final Project\\Datasets\\health counseling dataset\\train.csv')

In [13]:
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3512 entries, 0 to 3511
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Context   3512 non-null   object
 1   Response  3508 non-null   object
dtypes: object(2)
memory usage: 55.0+ KB


In [15]:
df.isnull().sum()

Context     0
Response    4
dtype: int64

In [16]:
#Removing null values
df.dropna(inplace=True)

In [18]:
df.to_csv('D:\\ITI\\9 Months Training AI\\NLP\\Final Project\\Datasets\\health counseling dataset\\train.csv', index=False)	

In [10]:
#max length of the question and answer
max_question_length = df['Context'].apply(lambda x: len(x.split())).max()
max_answer_length = df['Response'].apply(lambda x: len(x.split())).max()
print(f'Max question length: {max_question_length}')
print(f'Max answer length: {max_answer_length}')

Max question length: 526
Max answer length: 939


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

embedding = model.encode("Machine learning is amazing")

print(embedding.shape)

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|████

(768,)


In [5]:
embedding = model.encode("what is machine learning?")


In [1]:
import onnxruntime
print(onnxruntime.__version__)

1.22.0


In [7]:
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv
load_dotenv()
client = QdrantClient(
    url=os.getenv("QDRANT_CLUSTER_ENDPOINT"),
	api_key=os.getenv("QDRANT_API_KEY")
)

print(client.get_collections())

collections=[]


In [8]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="health-counseling-dataset",
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

True

In [9]:
print(client.get_collections())

collections=[CollectionDescription(name='health-counseling-dataset')]


In [10]:
documents = [
    "Machine learning is a branch of AI.",
    "Deep learning uses neural networks.",
    "Qdrant is a vector database."
]

embeddings = model.encode(documents)
print(embeddings.shape)

(3, 768)


In [ ]:
texts = (
    df["Context"].astype(str) + " " + df["Response"].astype(str)
).tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True  # optional but good for cosine
)


Batches: 100%|██████████| 110/110 [03:14<00:00,  1.77s/it]


IndexError: index 3508 is out of bounds for axis 0 with size 3508

In [32]:
def chunk_words(text, chunk_size=100, overlap=10):
    words = text.split()
    if not words:
        return []
        
    chunks = []
    # Guardrail against infinite loops
    step = max(1, chunk_size - overlap)

    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        
        # Instead of dropping small tail chunks, append to the last chunk if possible
        if len(chunk) < 20 and chunks:
            chunks[-1] = chunks[-1] + " " + " ".join(chunk)
        else:
            chunks.append(" ".join(chunk))

    return chunks

In [33]:
import time
import pandas as pd
from qdrant_client.models import PointStruct

# Configuration
batch_size = 4
global_id = 0  
qdrant_upload_batch_size = 64 # Ideal size for network payloads

for start in range(0, len(df), batch_size):
    end = min(start + batch_size, len(df))
    batch_df = df.iloc[start:end]

    points = []
    texts = []
    meta_map = []  

    # 1. Create chunks with extra tracking metadata
    for idx, row in batch_df.iterrows():
        full_text = f"{row['Context']} {row['Response']}"
        chunks = chunk_words(full_text, chunk_size=100, overlap=10)

        for chunk_idx, chunk in enumerate(chunks):
            texts.append(chunk)
            # Store a tuple of row data and chunk metadata
            meta_map.append({
                "row": row,
                "chunk_idx": chunk_idx,
                "total_chunks": len(chunks),
                "original_df_index": idx
            })

    if not texts:
        continue

    # 2. Embed chunks
    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # 3. Build points with structured payloads
    for i, embedding in enumerate(embeddings):
        meta = meta_map[i]
        row = meta["row"]

        points.append(
            PointStruct(
                id=global_id,
                vector=embedding.tolist(),
                payload={
                    "context": str(row["Context"])[:1000],
                    "response": str(row["Response"])[:1500],
                    "chunk": texts[i],
                    # Enhanced Metadata for filtering later
                    "chunk_idx": meta["chunk_idx"],
                    "total_chunks": meta["total_chunks"],
                    "source_row_id": int(meta["original_df_index"]) 
                }
            )
        )
        global_id += 1

    # 4. Upload safely using Qdrant's optimized upload utility
    try:
        client.upload_points(
            collection_name="health-counseling-dataset",
            points=points,
            batch_size=qdrant_upload_batch_size,
            parallel=2, # Uses 2 threads behind the scenes
            max_retries=3
        )
    except Exception as e:
        print(f"Failed to upload batch starting at row {start}: {e}")
        # Here you might want to log failed IDs to a file so you don't lose them

Failed to upload batch starting at row 788: Thread unexpectedly terminated
Failed to upload batch starting at row 1240: Thread unexpectedly terminated
Failed to upload batch starting at row 2144: Thread unexpectedly terminated
Failed to upload batch starting at row 2172: Thread unexpectedly terminated
Failed to upload batch starting at row 2304: Thread unexpectedly terminated
Failed to upload batch starting at row 3060: Thread unexpectedly terminated
Failed to upload batch starting at row 3140: Thread unexpectedly terminated
Failed to upload batch starting at row 3400: Thread unexpectedly terminated


In [24]:
from qdrant_client.models import PointStruct
import time

batch_size = 4

for start in range(0, len(df), batch_size):

    end = min(start + batch_size, len(df))
    batch_df = df.iloc[start:end]

    batch_texts = (
        batch_df["Context"].astype(str) + " " + batch_df["Response"].astype(str)
    ).tolist()

    batch_embeddings = model.encode(
        batch_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    points = []

    for i in range(len(batch_df)):
        row = batch_df.iloc[i]

        points.append(
            PointStruct(
                id=start + i,
                vector=batch_embeddings[i].tolist(),
                payload={
                    "context": row["Context"][:1000],   # truncate (IMPORTANT)
                    "response": row["Response"][:1500]  # prevent huge payload
                }
            )
        )

    # retry logic (VERY important for cloud)
    for attempt in range(3):
        try:
            client.upsert(
                collection_name="health-counseling-dataset",
                points=points
            )
            break
        except Exception as e:
            print(f"Retry {attempt+1}: {e}")
            time.sleep(3)

Retry 1: The read operation timed out


In [30]:
client.get_collection("health-counseling-dataset")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=255, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None

In [26]:
from qdrant_client.models import Filter

points, next_page_offset = client.scroll(
    collection_name="health-counseling-dataset",
    limit=10,   # number of samples
    with_payload=True,
    with_vectors=False
)

for p in points:
    print("ID:", p.id)
    print("Payload:", p.payload)
    print("-" * 40)

ID: 0
Payload: {'context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that

In [31]:
from qdrant_client.models import FilterSelector
client.delete(
    collection_name="health-counseling-dataset",
    points_selector=FilterSelector(
        filter=Filter()
    )
)

UpdateResult(operation_id=944, status=<UpdateStatus.COMPLETED: 'completed'>)

In [3]:
df

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
3503,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...
3504,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...
3505,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit..."
3506,I think adult life is making him depressed and...,How do you help yourself to believe you requir...
